Zuveir Jameer 19 March 2025

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
# converts each image into a PyTorch tensor.
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform) #loads the training set of 50,000 images.
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform) #loads the test set of 10,000 images

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) #creates training batches of images
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) #creates testing/validation batches of images

images, labels = next(iter(train_loader))

print(images.shape)   # torch.Size([64, 3, 32, 32])
print(labels.shape)   # torch.Size([64])

torch.Size([64, 3, 32, 32])
torch.Size([64])


In [ ]:
# -----------------------------
# Intermediate block
# x' = a1*C1(x) + a2*C2(x) + ... + aL*CL(x)
# -----------------------------
class IntermediateBlock(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_sizes):
        super().__init__()

        # Convolution branches. kernel_sizes = [1, 3, 5] and this creates 3 parallel convolution layers: 1x1; 3x3; 5x5
        self.branches = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=k, padding=k // 2) # padding=k // 2 keeps the output height and width the same as the input. 1x1 uses padding 0, 3x3 uses padding 1, 5x5 uses padding 2. That matters because all branch outputs must have the same shape before you add them together.

            for k in kernel_sizes
        ])

        # Creates the layer that computes the weight vector a.
        # in_channels = number of channels in x,
        # output size = number of convolution branches L
        self.fc = nn.Linear(in_channels, len(kernel_sizes))

    #Forward pass of IntermediateBlock
    def forward(self, x):
        # x: [batch, c, h, w]

        # If x has shape  [batch, c, h, w] then averaging over h and w gives [batch, c].
        # The average each channel and store it in vector m.
        # Channel-wise average -> m: [batch, c]. Example: input shape [64, 3, 32, 32], output m shape [64, 3]
        # Each image becomes a vector containing one average value per channel.
        m = x.mean(dim=(2, 3))

        # Compute the branch weights
        # a: [batch, L] where L is the number of branches. Each image in the batch gets its own learned combination of branches.
        a = self.fc(m)

        # Optional to normalized weights:
        # a = torch.softmax(a, dim=1)

        # Compute outputs from all branches
        # combines each  branch(x)  output tensor into one tensor of shape [batch, out_channels, h, w]
        # Branch outputs: [batch, L, out_channels, h, w]
        branch_outputs = torch.stack([branch(x) for branch in self.branches], dim=1)

        # Reshape the weights a so it can multiply each branch output
        # This is done so PyTorch can broadcast the weights across channels, height, width. So each branch output gets multiplied by its corresponding scalar weight.
        # Before a has shape [batch, L]
        # After a has shape [batch, L, 1, 1, 1]
        a = a.view(a.size(0), a.size(1), 1, 1, 1)

        # Weighted sum over branches -> [batch, out_channels, h, w]
        # x′= a1​C1​(x) + a2​C2​(x) +....+ aL​CL​(x)
        # Shape: [batch, out_channels, h, w]. This is the block's output.
        x_prime = (a * branch_outputs).sum(dim=1)

        return x_prime


# -----------------------------
# Output block
# averages each channel, then uses FC layers to output logits
# -----------------------------
class OutputBlock(nn.Module):
    def __init__(self, in_channels, num_classes=10, hidden_dims=None):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = []

        layers = []
        input_dim = in_channels

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, num_classes))
        self.classifier = nn.Sequential(*layers)

    def forward(self, x):
        # x: [batch, c, h, w]
        m = x.mean(dim=(2, 3))   # [batch, c]
        logits = self.classifier(m)
        return logits


# -----------------------------
# Full network: combines several intermediate blocks and one output block.
# -----------------------------
class BasicNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = IntermediateBlock(in_channels=3, out_channels=32, kernel_sizes=[1, 3, 5]) #RGB input with 3 channels and outputs 32 channels
        self.block2 = IntermediateBlock(in_channels=32, out_channels=64, kernel_sizes=[1, 3, 5]) #takes 32 channels and outputs 64 channels
        self.block3 = IntermediateBlock(in_channels=64, out_channels=128, kernel_sizes=[1, 3, 5]) #takes 64 channels and outputs 128 channels

        self.output_block = OutputBlock(in_channels=128, num_classes=10, hidden_dims=[64])

    def forward(self, x):
        x = self.block1(x)
        x = torch.relu(x)

        x = self.block2(x)
        x = torch.relu(x)

        x = self.block3(x)
        x = torch.relu(x)

        logits = self.output_block(x)
        return logits


# -----------------------------
# Train / test
# -----------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return 100.0 * correct / total


# -----------------------------
# Run
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BasicNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_accuracy = evaluate(model, test_loader, device)

    print(f"Epoch {epoch + 1}/{epochs} | Loss: {train_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%")

Epoch 1/20 | Loss: 2.0587 | Test Accuracy: 27.89%
Epoch 2/20 | Loss: 1.7989 | Test Accuracy: 32.59%
Epoch 3/20 | Loss: 1.6477 | Test Accuracy: 42.47%
Epoch 4/20 | Loss: 1.5075 | Test Accuracy: 47.65%
Epoch 5/20 | Loss: 1.3943 | Test Accuracy: 50.48%
Epoch 6/20 | Loss: 1.3016 | Test Accuracy: 53.29%
Epoch 7/20 | Loss: 1.2246 | Test Accuracy: 55.59%
Epoch 8/20 | Loss: 1.1580 | Test Accuracy: 56.81%
Epoch 9/20 | Loss: 1.1040 | Test Accuracy: 60.63%
Epoch 10/20 | Loss: 1.0555 | Test Accuracy: 62.42%
Epoch 11/20 | Loss: 1.0023 | Test Accuracy: 63.57%
Epoch 12/20 | Loss: 0.9656 | Test Accuracy: 65.37%
Epoch 13/20 | Loss: 0.9211 | Test Accuracy: 66.34%
Epoch 14/20 | Loss: 0.8923 | Test Accuracy: 66.57%
Epoch 15/20 | Loss: 0.8639 | Test Accuracy: 68.11%
Epoch 16/20 | Loss: 0.8321 | Test Accuracy: 69.56%
Epoch 17/20 | Loss: 0.8097 | Test Accuracy: 69.02%
Epoch 18/20 | Loss: 0.7811 | Test Accuracy: 69.91%
Epoch 19/20 | Loss: 0.7536 | Test Accuracy: 71.55%
Epoch 20/20 | Loss: 0.7391 | Test Accura